# Pruning itératif de Mistral-7B + récupération (LoRA / distillation)

Notebook **reprenable** : tout est écrit sur disque (`history.csv`, `palier_X/fp16_pruned`, `palier_X/adapter`).
Après un crash ou un redémarrage du kernel, il suffit de **relancer toutes les cellules** : les étapes déjà faites sont sautées.

Corrections par rapport à la version précédente :
- OOM : batch 1 + accumulation, gradient checkpointing, optimiseur 8 bits, `max_length` limité, `expandable_segments`, nettoyage mémoire entre chaque étape
- Reprise : pruning sauté si `fp16_pruned` est complet, adapter rechargé s'il existe déjà
- Historique persistant (CSV) : plus de `history_df` perdu au redémarrage
- Graphique tolérant aux points manquants
- Adapter **sauvegardé sur disque avant** tout push, push via `upload_folder` (sans recharger le modèle)
- Distillation optionnelle (professeur = Mistral d'origine en NF4)

## 1. Configuration

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # AVANT d'importer torch
os.environ["TOKENIZERS_PARALLELISM"] = "false"

MODEL_ID   = "mistralai/Mistral-7B-v0.1"
OUTPUT_DIR = "/kaggle/working/mistral-pruned-iterative"
PALIERS    = [0.20, 0.40]          # sparsite cible cumulative par palier

# --- Recuperation ---
STEPS_PER_PALIER = 75
MAX_LEN          = 512             # baisse si OOM (384, 256)
BATCH_SIZE       = 1
GRAD_ACCUM       = 8
LEARNING_RATE    = 1e-4

USE_DISTILLATION = True            # False = fine-tuning classique
KD_ALPHA         = 0.5             # poids de la distillation (1-alpha = perte classique)
KD_TEMPERATURE   = 2.0

# --- Hugging Face ---
HF_USER = "TON_PSEUDO"             # <-- a remplacer
PRIVATE_REPOS = True

os.makedirs(OUTPUT_DIR, exist_ok=True)
HISTORY_CSV = f"{OUTPUT_DIR}/history.csv"

## 2. Imports, authentification, utilitaires

In [ ]:
import gc, json, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from safetensors import safe_open
from datasets import Dataset
from huggingface_hub import HfApi, login
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          DataCollatorForLanguageModeling, Trainer, TrainingArguments)

# Token HF (Kaggle: Add-ons -> Secrets -> HF_TOKEN, droits d'ecriture)
try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
    print("Connecte a Hugging Face.")
except Exception as e:
    print(f"Pas de connexion HF ({e}). Le push sera impossible tant que ce n'est pas regle.")

def cleanup():
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated() / 2**30
        print(f"   [VRAM allouee apres nettoyage : {used:.2f} Gio]")

def palier_dir(i):    return f"{OUTPUT_DIR}/palier_{i}"
def fp16_dir(i):      return f"{palier_dir(i)}/fp16_pruned"
def adapter_dir(i):   return f"{palier_dir(i)}/adapter"

def disk_report():
    total, used, free = shutil.disk_usage("/kaggle/working")
    print(f"Disque /kaggle/working : {used/2**30:.1f} Gio utilises, {free/2**30:.1f} Gio libres")
    print("(limite Kaggle ~20 Gio : un modele fp16 pese ~14 Gio, pousse-le sur le Hub puis supprime l'ancien palier si besoin)")

# ---------- Historique persistant ----------
HIST_COLS = ["palier", "phase", "accuracy", "real_sparsity", "target_sparsity"]

def load_history():
    if os.path.exists(HISTORY_CSV):
        return pd.read_csv(HISTORY_CSV)
    return pd.DataFrame(columns=HIST_COLS)

history_df = load_history()

def has(palier, phase):
    return ((history_df["palier"] == palier) & (history_df["phase"] == phase)).any()

def record(palier, phase, accuracy, real_sparsity, target_sparsity):
    global history_df
    row = pd.DataFrame([[palier, phase, accuracy, real_sparsity, target_sparsity]], columns=HIST_COLS)
    history_df = pd.concat([history_df, row], ignore_index=True)
    history_df = history_df.drop_duplicates(["palier", "phase"], keep="last")
    history_df.to_csv(HISTORY_CSV, index=False)
    print(f"   -> historique : palier {palier} | {phase} | acc={accuracy:.2%} | sparsite={real_sparsity:.2%}")

print(history_df)
disk_report()

## 3. Données et évaluation  ⚠️ à adapter

Je n'ai pas ton jeu de données ni ta métrique d'accuracy : **remplis `train_texts` et `eval_examples`** (ou remplace `evaluate`
par ta fonction). `eval_examples` = liste de `{"prompt": ..., "answer": ...}` ; la réponse est jugée correcte si la génération
commence par `answer`.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# ====== A REMPLIR ======
train_texts   = []   # ex: ["Question: ... Reponse: ...", ...]
eval_examples = []   # ex: [{"prompt": "Question: ... Reponse:", "answer": "B"}, ...]
# =======================

assert len(train_texts) > 0 and len(eval_examples) > 0, \
    "Remplis train_texts et eval_examples (ou remplace evaluate()/train_dataset par ton propre code)."

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

train_dataset = Dataset.from_dict({"text": train_texts}).map(tokenize, batched=True, remove_columns=["text"])

@torch.no_grad()
def evaluate(model, examples=None):
    examples = examples if examples is not None else eval_examples
    model.eval()
    correct = 0
    for ex in examples:
        enc = tokenizer(ex["prompt"], return_tensors="pt", truncation=True, max_length=MAX_LEN).to(model.device)
        out = model.generate(**enc, max_new_tokens=8, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id, use_cache=True)
        txt = tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
        correct += txt.strip().lower().startswith(ex["answer"].strip().lower())
    return correct / len(examples)

## 4. Pruning (CPU) et lecture de la sparsité sur disque

In [ ]:
def prunable_linears(model):
    # 32 couches x 7 projections = 224 modules pour Mistral-7B
    return [(n, m) for n, m in model.model.layers.named_modules() if isinstance(m, torch.nn.Linear)]

@torch.no_grad()
def prune_model(model, target):
    # Magnitude pruning par couche : amene chaque matrice a `target` de zeros (les zeros existants comptent).
    layers = prunable_linears(model)
    for name, lin in layers:
        w = lin.weight.data
        k = int(w.numel() * target)
        if k < 1:
            continue
        thr = torch.kthvalue(w.abs().flatten().float(), k).values
        if thr.item() > 0:
            w[w.abs() <= thr.to(w.dtype)] = 0
    print(f"   {len(layers)} couches prunees")

def is_complete(path):
    # Un dossier est complet si l'index existe et que tous les shards listes sont presents.
    idx = f"{path}/model.safetensors.index.json"
    if os.path.exists(idx):
        shards = set(json.load(open(idx))["weight_map"].values())
        return all(os.path.exists(f"{path}/{s}") for s in shards)
    return os.path.exists(f"{path}/model.safetensors")

def sparsity_from_disk(path):
    zeros = total = 0
    files = [f for f in os.listdir(path) if f.endswith(".safetensors")]
    for f in files:
        with safe_open(f"{path}/{f}", framework="pt") as sf:
            for key in sf.keys():
                if ".layers." in key and key.endswith("_proj.weight"):
                    t = sf.get_tensor(key)
                    zeros += (t == 0).sum().item()
                    total += t.numel()
    return zeros / total

def ensure_pruned(i, target):
    out = fp16_dir(i)
    if is_complete(out):
        print(f"[pruning] palier {i} deja present -> {out}, on saute.")
        return
    src = MODEL_ID if i == 1 else fp16_dir(i - 1)
    print(f"[pruning] palier {i} : {src} -> {target:.0%}")
    model = AutoModelForCausalLM.from_pretrained(src, torch_dtype=torch.float16,
                                                 device_map="cpu", low_cpu_mem_usage=True)
    prune_model(model, target)
    os.makedirs(out, exist_ok=True)
    model.save_pretrained(out, max_shard_size="2GB", safe_serialization=True)
    tokenizer.save_pretrained(out)
    del model
    gc.collect()
    print(f"   sauvegarde -> {out}")
    disk_report()

## 5. Chargement NF4 + LoRA

In [ ]:
BNB = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)

def load_nf4(path):
    return AutoModelForCausalLM.from_pretrained(path, quantization_config=BNB,
                                                device_map={"": 0}, torch_dtype=torch.float16)

def attach_lora(model):
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                     target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                     "gate_proj", "up_proj", "down_proj"])
    return get_peft_model(model, cfg)

## 6. Trainer (fine-tuning classique ou distillation)

In [ ]:
class DistillTrainer(Trainer):
    # Si teacher=None : perte classique. Sinon : alpha * KL(prof || eleve) * T^2 + (1-alpha) * perte classique.
    def __init__(self, *args, teacher=None, temperature=2.0, alpha=0.5, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher, self.T, self.alpha = teacher, temperature, alpha

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        out = model(**inputs)
        if self.teacher is None:
            return (out.loss, out) if return_outputs else out.loss

        with torch.no_grad():
            t_logits = self.teacher(input_ids=inputs["input_ids"],
                                    attention_mask=inputs["attention_mask"]).logits
        mask = inputs["labels"][:, 1:] != -100
        s = out.logits[:, :-1][mask].float() / self.T
        t = t_logits[:, :-1][mask].float() / self.T
        kd = F.kl_div(F.log_softmax(s, -1), F.softmax(t, -1), reduction="batchmean") * (self.T ** 2)
        loss = self.alpha * kd + (1 - self.alpha) * out.loss
        return (loss, out) if return_outputs else loss

def recovery_finetune(peft_model, out_dir, teacher=None):
    args = TrainingArguments(
        output_dir=out_dir,
        max_steps=STEPS_PER_PALIER,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="paged_adamw_8bit",
        learning_rate=LEARNING_RATE,
        warmup_steps=5,
        fp16=True,
        logging_steps=10,
        save_strategy="no",
        report_to="none",
        remove_unused_columns=False,
        label_names=["labels"],
    )
    trainer = DistillTrainer(
        model=peft_model, args=args, train_dataset=train_dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
        processing_class=tokenizer,
        teacher=teacher, temperature=KD_TEMPERATURE, alpha=KD_ALPHA,
    )
    peft_model.config.use_cache = False
    trainer.train()
    peft_model.config.use_cache = True
    return trainer

## 7. Étapes d'un palier

Chaque étape vit dans sa propre fonction : à la sortie, ses variables locales (modèle, trainer, optimiseur) sont libérées,
ce qui évite l'accumulation en VRAM qui a causé l'OOM.

In [ ]:
def step_baseline():
    print("[baseline] evaluation du modele d'origine (NF4)")
    model = load_nf4(MODEL_ID)
    acc = evaluate(model)
    record(0, "baseline", acc, 0.0, 0.0)

def step_eval_pruned(i, target, sparsity):
    print(f"[palier {i}] evaluation apres pruning, avant recuperation")
    model = load_nf4(fp16_dir(i))
    acc = evaluate(model)
    record(i, "pruned_before_recovery", acc, sparsity, target)

def step_recover(i, target, sparsity):
    a_dir = adapter_dir(i)

    # Adapter deja sauvegarde : on l'evalue sans re-entrainer
    if os.path.exists(f"{a_dir}/adapter_config.json"):
        print(f"[palier {i}] adapter existant -> evaluation seule")
        base = load_nf4(fp16_dir(i))
        peft_model = PeftModel.from_pretrained(base, a_dir)
        acc = evaluate(peft_model)
        record(i, "recovered", acc, sparsity, target)
        return

    mode = "distillation" if USE_DISTILLATION else "fine-tuning classique"
    print(f"[palier {i}] recuperation ({mode}, {STEPS_PER_PALIER} steps)")

    teacher = None
    if USE_DISTILLATION:
        teacher = load_nf4(MODEL_ID)          # modele d'origine, gele
        teacher.eval()
        for p in teacher.parameters():
            p.requires_grad = False

    student = attach_lora(load_nf4(fp16_dir(i)))
    student.print_trainable_parameters()

    trainer = recovery_finetune(student, f"{palier_dir(i)}/trainer_out", teacher=teacher)

    # Sauvegarde sur disque AVANT l'evaluation et avant tout push
    student.save_pretrained(a_dir)
    tokenizer.save_pretrained(a_dir)
    print(f"   adapter sauvegarde -> {a_dir}")

    acc = evaluate(student)
    record(i, "recovered", acc, sparsity, target)

## 8. Boucle principale (reprenable)

In [ ]:
if not has(0, "baseline"):
    step_baseline()
    cleanup()

for i, target in enumerate(PALIERS, start=1):
    print("=" * 70)
    print(f"PALIER {i}/{len(PALIERS)} - cible {target:.0%}")
    print("=" * 70)

    ensure_pruned(i, target)
    sparsity = sparsity_from_disk(fp16_dir(i))
    print(f"   sparsite reelle : {sparsity:.2%}")

    if not has(i, "pruned_before_recovery"):
        step_eval_pruned(i, target, sparsity)
        cleanup()

    if not has(i, "recovered"):
        step_recover(i, target, sparsity)
        cleanup()
    else:
        print(f"[palier {i}] deja termine, on saute.")

print("\nTermine.")
print(history_df.sort_values(["palier", "phase"]))

## 9. Courbe accuracy avant / après récupération

In [ ]:
def get_value(df, palier, phase, column):
    row = df[(df["palier"] == palier) & (df["phase"] == phase)]
    return row[column].values[0] if len(row) else np.nan

fig, ax = plt.subplots(figsize=(10, 6))

paliers_pruning = sorted(p for p in history_df["palier"].unique() if p > 0)

before_acc = [get_value(history_df, p, "pruned_before_recovery", "accuracy") for p in paliers_pruning]
after_acc  = [get_value(history_df, p, "recovered", "accuracy") for p in paliers_pruning]
sparsities = [get_value(history_df, p, "pruned_before_recovery", "real_sparsity") for p in paliers_pruning]

baseline_acc = get_value(history_df, 0, "baseline", "accuracy")

x_labels = ["baseline"] + [
    f"palier {p}\n({s:.0%} sparse)" if not np.isnan(s) else f"palier {p}"
    for p, s in zip(paliers_pruning, sparsities)
]
y_before = [baseline_acc] + before_acc
y_after  = [baseline_acc] + after_acc
x = list(range(len(x_labels)))

ax.plot(x, y_before, marker="o", color="#FF6B6B", label="Avant recuperation", linewidth=2)
ax.plot(x, y_after,  marker="o", color="#00D4AA", label="Apres recuperation", linewidth=2)
ax.set_xticks(x)
ax.set_xticklabels(x_labels)
ax.set_ylabel("Accuracy")
ax.set_title("Pruning iteratif - accuracy avant/apres recuperation, par palier")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/iterative_pruning_curve.png", dpi=150)
plt.show()
print(f"Courbe sauvegardee -> {OUTPUT_DIR}/iterative_pruning_curve.png")

## 10. Push sur Hugging Face

`upload_folder` envoie les fichiers tels quels : pas de rechargement du modèle, donc pas de `NameError` ni de RAM consommée.
L'adapter n'est utilisable qu'avec le modèle élagué du **même palier** : pousse les deux si tu veux tout recharger depuis le Hub.

In [ ]:
PUSH_PALIERS      = [1]      # ex: [1], [2] ou [1, 2]
PUSH_ADAPTER      = True     # leger (quelques dizaines de Mo)
PUSH_PRUNED_MODEL = True     # lourd (~14 Gio par palier)

api = HfApi()
for i in PUSH_PALIERS:
    pct = int(round(PALIERS[i - 1] * 100))
    if PUSH_ADAPTER:
        if os.path.exists(adapter_dir(i)):
            repo = f"{HF_USER}/mistral-pruned-{pct}-adapter"
            api.create_repo(repo, private=PRIVATE_REPOS, exist_ok=True)
            api.upload_folder(folder_path=adapter_dir(i), repo_id=repo)
            print(f"Adapter pousse -> https://huggingface.co/{repo}")
        else:
            print(f"Pas d'adapter pour le palier {i} (non entraine ou non sauvegarde).")
    if PUSH_PRUNED_MODEL:
        if is_complete(fp16_dir(i)):
            repo = f"{HF_USER}/mistral-pruned-{pct}"
            api.create_repo(repo, private=PRIVATE_REPOS, exist_ok=True)
            api.upload_folder(folder_path=fp16_dir(i), repo_id=repo)
            print(f"Modele pousse -> https://huggingface.co/{repo}")
        else:
            print(f"Modele du palier {i} incomplet ou absent.")

In [28]:
import os, re, json
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi

login(token=UserSecretsClient().get_secret("hf_write"))

ADAPTER_DIR = "/kaggle/working/mistral-pruned-iterative/palier_1/adapter"
REPO        = "Falcot922/mistral-pruned-20-adapter"
BASE_REPO   = "Falcot922/mistral-pruned-20"    # repo du modele elague du palier 1

# 1. README : retire la ligne base_model (chemin local invalide)
readme = f"{ADAPTER_DIR}/README.md"
txt = open(readme, encoding="utf-8").read()
txt = re.sub(r"(?m)^base_model:.*\n", "", txt, count=1)
open(readme, "w", encoding="utf-8").write(txt)

# 2. adapter_config.json : pointe vers le repo du modele elague, pas vers un chemin Kaggle
cfg_path = f"{ADAPTER_DIR}/adapter_config.json"
cfg = json.load(open(cfg_path))
print("avant :", cfg["base_model_name_or_path"])
cfg["base_model_name_or_path"] = BASE_REPO
json.dump(cfg, open(cfg_path, "w"), indent=2)

# 3. Push
api = HfApi()
api.create_repo(REPO, private=True, exist_ok=True)
api.upload_folder(folder_path=ADAPTER_DIR, repo_id=REPO)
print(f"Pousse -> https://huggingface.co/{REPO}")

avant : /kaggle/working/mistral-pruned-iterative/palier_1/fp16_pruned


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pousse -> https://huggingface.co/Falcot922/mistral-pruned-20-adapter
